# 第 11 讲：整数规划、0-1 规划与多目标优化

> 用枚举法实现小规模 0-1 规划，并展示多目标权衡。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-11"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

In [ ]:
projects = pd.DataFrame({
    "project": ["P1", "P2", "P3", "P4", "P5"],
    "benefit": [16, 22, 18, 12, 20],
    "cost": [8, 11, 9, 6, 10],
    "risk": [5, 7, 4, 3, 6],
})
budget = 24
rows = []
for bits in itertools.product([0, 1], repeat=len(projects)):
    chosen = np.array(bits)
    cost = (chosen * projects["cost"]).sum()
    if cost <= budget:
        rows.append({
            "selection": "".join(map(str, bits)),
            "benefit": (chosen * projects["benefit"]).sum(),
            "cost": cost,
            "risk": (chosen * projects["risk"]).sum(),
        })
solutions = pd.DataFrame(rows)
best = solutions.sort_values(["benefit", "risk"], ascending=[False, True]).head(1)
solutions.to_csv(OUTPUT_ROOT / "binary_solutions.csv", index=False)
best

In [ ]:
weights = np.linspace(0, 1, 21)
frontier = []
for w in weights:
    tmp = solutions.copy()
    tmp["score"] = w * tmp["benefit"] - (1 - w) * tmp["risk"]
    row = tmp.sort_values("score", ascending=False).iloc[0]
    frontier.append({"benefit_weight": w, "selection": row["selection"], "benefit": row["benefit"], "risk": row["risk"]})
frontier = pd.DataFrame(frontier)
frontier.to_csv(OUTPUT_ROOT / "multiobjective_tradeoff.csv", index=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(solutions["risk"], solutions["benefit"], alpha=0.5, label="feasible")
ax.plot(frontier["risk"], frontier["benefit"], "r-o", label="weighted choices")
ax.set_xlabel("Risk")
ax.set_ylabel("Benefit")
ax.set_title("Benefit-risk tradeoff")
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_ROOT / "multiobjective_tradeoff.png", dpi=160)
frontier.head()